# Notebook: BCI_22_CarDet_Crit_Entrada
*********************************************************************************

## Informacion del Notebook

### Encabezado
**************************************************************************
* Nombre: BCI_22_CarDet_Crit_Entrada.ipynb
* Ruta: https://adb-5512273708018582.2.azuredatabricks.net/?o=5512273708018582#notebook/2997520011901624
* Autor: Gabriel Martinez (SimpleData) - Ing. SW BCI: Jonatan Cancino
* Fecha: 12/08/2022
* Descripcion: Evaluacion criterios de entrada del periodo actual
* Documentacion:
***************************************************************************

### Mantenciones
**************************************************************************
#### Mantención Nro: 1
* Autor: Gagriel Martinez (SimpleData) - Ing. SW BCI: Jonatan Cancino
* Fecha: 10/02/2025 
* Descripción: Se cambia el Delete por Truncate al momento de eliminar los datos de la tabla tbl_cd_cartdet_crit_ent_crit     
***************************************************************************

### Tablas Entrada y Salida
**************************************************************************
#### Tablas Entrada: 
* {base_silver_x}.tbl_cd_cartdet_crit_ent_ope_eval 
***************************************************************************
#### Tablas Salida: 
* {base_silver_x}.tbl_cd_cartdet_crit_ent_crit
***************************************************************************


## Carga Dependencias

### Setea Parámetros

In [0]:

dbutils.widgets.text("fecha_w","","01-Fecha:")
dbutils.widgets.text("bd_silver_w","","03-Nombre BD Silver:")


fecha_x = dbutils.widgets.get("fecha_w") 
base_silver_x = dbutils.widgets.get("bd_silver_w")

spark.conf.set("bci.fecha", fecha_x)
spark.conf.set("bci.dbnamesilver", base_silver_x)

print(f"Fecha de Proceso actual: [fecha_x] {fecha_x}")
print(f"Nombre BD Silver: [base_silver_x] {base_silver_x}")


### Carga funciones comunes

In [0]:
%run "./Funciones_Comunes"

### Valida parámetros

In [0]:
valida_parametro(fecha_x)

In [0]:
valida_parametro(base_silver_x)

In [0]:
# Calcula periodo en base a la fecha
periodo_x=fecha_x[:6]

print(f"Periodo: [periodo_x] {periodo_x}")

## INICIO Proceso extraccion y transformacion
--------------------------------------
- Por cada fuente que se utilice se debe:
     - Titulo: generar un titulo generico, con nombre fuente, y descripcion del proposito de la extraccion
     - Extraer: para el periodo, o rango de fecha que se necesita la iformacion. Debe tener el prefijo tmp_EXT_{nombrefuente}
     - Transformar: generar la informacion necesaria para la salida final. Se pueden generar mas de una tabla temporal para llegar al resultado final. Debe tener el prefijo tmp_RES_{nombre}_correlativo


### Parametrizacion
---
* define y asigna valores a los parametros


In [0]:
p_cod_seg_gru='G'
p_crit_det = 1,7,8,9,10,11,12,13
p_periodo_evaluacion='p_actual'

print(f"p_cod_seg_gru: {p_cod_seg_gru}")
print(f"p_crit_det: {p_crit_det}")
print(f"p_periodo_evaluacion: {p_periodo_evaluacion}")


### Extrae Evaluaciones  (riesgobdu_silver_db.tbl_cartdet_crit_ent_ope_eval)
--------------------------------------
- Se extraen todas las evaluaciones positivas 

In [0]:
paso_query10 = f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_EXT_tbl_cartdet_crit_ent_ope_eval AS
SELECT 
        b.periodo_cierre,
        b.fecha_cierre,
        b.tipo_proceso,
        b.segmento,
        b.operacion,
        b.tipo_operacion,
        b.sistema,
        b.rut_cliente,
        b.dv_rut_cliente,
        b.nombre_campo,
        b.valor_campo,
        b.condicion_regla,
        b.valor_regla,
        b.flag_resultado_regla,
        b.cod_evaluacion
FROM
  {base_silver_x}.tbl_cd_cartdet_crit_ent_ope_eval b
WHERE 
    b.fecha_cierre =   {fecha_x} 
AND b.periodo_cierre = {periodo_x}
AND b.flag_resultado_regla = 1
QUALIFY  ROW_NUMBER() OVER(PARTITION BY b.operacion, b.sistema, b.cod_evaluacion ORDER BY b.fecha_cierre DESC) =1
"""


In [0]:
sql_safe(paso_query10)

### Matriz con evaluaciones de entrada por operacion
--------------------------------------
- Genera matriz por operacion con indicador de evaluacion indicando si cumple o no cumple dicha evaluacion

In [0]:
paso_query20 = f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_matriz_cartdet_crit_ent_ope_eval AS
SELECT
     periodo_cierre
    ,fecha_cierre
    ,tipo_proceso
    ,operacion
    ,sistema
    ,rut_cliente
    ,dv_rut_cliente
    ,tipo_operacion
    ,segmento
    ,MAX(CASE WHEN IFNULL(A.cod_evaluacion,'XXX') = 'E01' THEN 1 ELSE 0 END) AS IND_E01
    ,MAX(CASE WHEN IFNULL(A.cod_evaluacion,'XXX') = 'E04' THEN 1 ELSE 0 END) AS IND_E04
    ,MAX(CASE WHEN IFNULL(A.cod_evaluacion,'XXX') = 'E05' THEN 1 ELSE 0 END) AS IND_E05
    ,MAX(CASE WHEN IFNULL(A.cod_evaluacion,'XXX') = 'E06' THEN 1 ELSE 0 END) AS IND_E06
    ,MAX(CASE WHEN IFNULL(A.cod_evaluacion,'XXX') = 'E07' THEN 1 ELSE 0 END) AS IND_E07
    ,MAX(CASE WHEN IFNULL(A.cod_evaluacion,'XXX') = 'E08' THEN 1 ELSE 0 END) AS IND_E08
    ,MAX(CASE WHEN IFNULL(A.cod_evaluacion,'XXX') = 'E09' THEN 1 ELSE 0 END) AS IND_E09
    ,MAX(CASE WHEN IFNULL(A.cod_evaluacion,'XXX') = 'N01' THEN 1 ELSE 0 END) AS IND_N01
    ,MAX(CASE WHEN IFNULL(A.cod_evaluacion,'XXX') = 'N02' THEN 1 ELSE 0 END) AS IND_N02
FROM
    tmp_EXT_tbl_cartdet_crit_ent_ope_eval A
GROUP BY 
   1,2,3,4,5,6,7,8,9
"""   

In [0]:
sql_safe(paso_query20)

### Deterioro operaciones de clientes individuales
--------------------------------------
- Genera registros para operaciones deterioradas individualmente
- Para todas las operaciones del cliente que cumplan E01=1

In [0]:
paso_query50 =  f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_D00_OPE_CRIT_EVAL_1 as
SELECT
        periodo_cierre          AS periodo_cierre,
        fecha_cierre            AS fecha_cierre,
        tipo_proceso            AS tipo_proceso,
        rut_cliente             AS rut_cliente,
        dv_rut_cliente          AS dv_rut_cliente,
        tipo_operacion          AS tipo_operacion,
        operacion               AS operacion,
        sistema                 AS sistema,
        segmento                AS segmento,
        1                       AS criterio_entrada,
        0                       AS origen_deterioro,
        fecha_cierre            AS fecha_entrada,
        'BCI_Individual'        AS grupo
FROM 
    tmp_RES_matriz_cartdet_crit_ent_ope_eval  A
WHERE
    IND_E01 = 1 AND IND_N01=0 AND IND_N02=0
"""


In [0]:
sql_safe(paso_query50)

### Deterioro operaciones con morosidad 
--------------------------------------
- Genera registros para operaciones grupalmente
- Operaciones que cumplen con: Operacion Grupal and E04 =1


In [0]:
paso_query55 =  f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_D00_OPE_CRIT_EVAL_2 as
SELECT
        A.periodo_cierre          AS periodo_cierre,
        A.fecha_cierre            AS fecha_cierre,
        A.tipo_proceso            AS tipo_proceso,
        A.rut_cliente             AS rut_cliente,
        A.dv_rut_cliente          AS dv_rut_cliente,
        A.tipo_operacion          AS tipo_operacion,
        A.operacion               AS operacion,
        A.sistema                 AS sistema,
        A.segmento                AS segmento,
        CASE 
          WHEN SUBSTRING(A.tipo_operacion,1,3) IN ('HIP','CAE') 
          THEN 8 
          ELSE 7
        END                     AS criterio_entrada,
        1                       AS origen_deterioro,
        A.fecha_cierre          AS fecha_entrada,
        'BCI_Grupal'            AS grupo
FROM 
    tmp_RES_matriz_cartdet_crit_ent_ope_eval  A
WHERE
    A.segmento= '{p_cod_seg_gru}'
AND A.IND_E04 = 1 AND IND_N01=0 AND IND_N02=0
"""


In [0]:
sql_safe(paso_query55)

### Deterioro operaciones renegociadas bci
--------------------------------------
- Genera registros para operaciones grupales
- Operaciones que cumplen con: Operacion Grupal and E05=1


In [0]:
paso_query60 =  f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_D00_OPE_CRIT_EVAL_3 as
SELECT
        A.periodo_cierre          AS periodo_cierre,
        A.fecha_cierre            AS fecha_cierre,
        A.tipo_proceso            AS tipo_proceso,
        A.rut_cliente             AS rut_cliente,
        A.dv_rut_cliente          AS dv_rut_cliente,
        A.tipo_operacion          AS tipo_operacion,
        A.operacion               AS operacion,
        A.sistema                 AS sistema,
        A.segmento                AS segmento,
        9                         AS criterio_entrada,
        3                         AS origen_deterioro,
        A.fecha_cierre            AS fecha_entrada,
        'BCI_Grupal'              AS grupo
FROM 
    tmp_RES_matriz_cartdet_crit_ent_ope_eval  A
WHERE
    A.segmento= '{p_cod_seg_gru}'
AND A.IND_E05=1 AND IND_N01=0 AND IND_N02=0
"""


In [0]:
sql_safe(paso_query60)

### Deterioro operaciones con curse bajo mora bci
--------------------------------------
- Genera registros para operaciones deterioradas grupalmente
- Operaciones que cumplen con: Operacion Grupal and E06=1


In [0]:
paso_query65 =  f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_D00_OPE_CRIT_EVAL_4 as
SELECT
        A.periodo_cierre          AS periodo_cierre,
        A.fecha_cierre            AS fecha_cierre,
        A.tipo_proceso            AS tipo_proceso,
        A.rut_cliente             AS rut_cliente,
        A.dv_rut_cliente          AS dv_rut_cliente,
        A.tipo_operacion          AS tipo_operacion,
        A.operacion               AS operacion,
        A.sistema                 AS sistema,
        A.segmento                AS segmento,
        10                         AS criterio_entrada,
        4                         AS origen_deterioro,
        A.fecha_cierre            AS fecha_entrada,
        'BCI_Grupal'              AS grupo
FROM 
    tmp_RES_matriz_cartdet_crit_ent_ope_eval  A
WHERE
    A.segmento= '{p_cod_seg_gru}'
AND A.IND_E06=1 AND IND_N01=0 AND IND_N02=0
"""


In [0]:
sql_safe(paso_query65)

### Deterioro operaciones clientes LIR
--------------------------------------
- Genera registros para operaciones deterioradas grupalmente
- Operaciones que cumplen con: Operacion Grupal and E07=1


In [0]:
paso_query70 =  f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_D00_OPE_CRIT_EVAL_5 as
SELECT
        A.periodo_cierre          AS periodo_cierre,
        A.fecha_cierre            AS fecha_cierre,
        A.tipo_proceso            AS tipo_proceso,
        A.rut_cliente             AS rut_cliente,
        A.dv_rut_cliente          AS dv_rut_cliente,
        A.tipo_operacion          AS tipo_operacion,
        A.operacion               AS operacion,
        A.sistema                 AS sistema,
        A.segmento                AS segmento,
        11                        AS criterio_entrada,
        2                         AS origen_deterioro,
        A.fecha_cierre            AS fecha_entrada,
        'BCI_Grupal'              AS grupo
FROM 
    tmp_RES_matriz_cartdet_crit_ent_ope_eval  A
WHERE
    A.segmento= '{p_cod_seg_gru}'
AND A.IND_E07=1 AND IND_N01=0 AND IND_N02=0
"""


In [0]:
sql_safe(paso_query70)

### Deterioro operaciones de clientes Informados x SSFF
--------------------------------------
- Genera registros para operaciones deterioradas grupalmente
- Operaciones que cumplen con: Operacion Grupal and E08=1


In [0]:
paso_query75 =  f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_D00_OPE_CRIT_EVAL_6 as
SELECT
        A.periodo_cierre          AS periodo_cierre,
        A.fecha_cierre            AS fecha_cierre,
        A.tipo_proceso            AS tipo_proceso,
        A.rut_cliente             AS rut_cliente,
        A.dv_rut_cliente          AS dv_rut_cliente,
        A.tipo_operacion          AS tipo_operacion,
        A.operacion               AS operacion,
        A.sistema                 AS sistema,
        A.segmento                AS segmento,
        12                        AS criterio_entrada,
        6                         AS origen_deterioro,
        A.fecha_cierre            AS fecha_entrada,
        'SSFF'                    AS grupo
FROM 
    tmp_RES_matriz_cartdet_crit_ent_ope_eval  A
WHERE
    A.segmento= '{p_cod_seg_gru}'
AND A.IND_E08=1 AND IND_N01=0 AND IND_N02=0
"""


In [0]:
sql_safe(paso_query75)

### Deterioro operaciones de clientes Informados x Factoring
--------------------------------------
- Genera registros para operaciones deterioradas grupalmente
- Operaciones que cumplen con: Operacion Grupal and E09=1


In [0]:
paso_query80 =  f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_D00_OPE_CRIT_EVAL_7 as
SELECT
        A.periodo_cierre          AS periodo_cierre,
        A.fecha_cierre            AS fecha_cierre,
        A.tipo_proceso            AS tipo_proceso,
        A.rut_cliente             AS rut_cliente,
        A.dv_rut_cliente          AS dv_rut_cliente,
        A.tipo_operacion          AS tipo_operacion,
        A.operacion               AS operacion,
        A.sistema                 AS sistema,
        A.segmento                AS segmento,
        13                        AS criterio_entrada,
        6                         AS origen_deterioro,
        A.fecha_cierre           AS fecha_entrada,
        'Factoring'              AS grupo
FROM 
    tmp_RES_matriz_cartdet_crit_ent_ope_eval  A
WHERE
    A.segmento= '{p_cod_seg_gru}'
AND A.IND_E09=1 AND IND_N01=0 AND IND_N02=0
"""


In [0]:
sql_safe(paso_query80)

### Salida Temporal a Nivel de Campo Evaludado (tmp_tbl_cartdet_crit_ent_crit)
------------------
* generar salida temporal a nivel de campo evaluado. 
* se registran todas las operaciones evaluadas


In [0]:

paso_query250 = f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_tbl_cartdet_crit_ent_crit AS
SELECT * FROM   tmp_RES_D00_OPE_CRIT_EVAL_1  
UNION 
SELECT * FROM   tmp_RES_D00_OPE_CRIT_EVAL_2  
UNION
SELECT * FROM   tmp_RES_D00_OPE_CRIT_EVAL_3  
UNION
SELECT * FROM   tmp_RES_D00_OPE_CRIT_EVAL_4 
UNION
SELECT * FROM   tmp_RES_D00_OPE_CRIT_EVAL_5 
UNION
SELECT * FROM   tmp_RES_D00_OPE_CRIT_EVAL_6 
UNION
SELECT * FROM   tmp_RES_D00_OPE_CRIT_EVAL_7 
"""  

In [0]:
sql_safe(paso_query250)

## Carga Tablas de Salidas
--------------------------------------
* carga resultados a tablas de salidas del notebook

### Carga Tabla Evaluacion 


#### Reproceso (Elimina registros en caso de reprocesos). Tabla no es historica.

In [0]:
paso_query300 = f""" TRUNCATE TABLE {base_silver_x}.tbl_cd_cartdet_crit_ent_crit """

In [0]:
sql_safe(paso_query300)

#### Inserta Registros tabla salida

In [0]:
paso_query310 = f"""
INSERT INTO {base_silver_x}.tbl_cd_cartdet_crit_ent_crit
SELECT 
    IFNULL(periodo_cierre,190001),
    IFNULL(fecha_cierre,19000101),
    IFNULL(tipo_proceso,' '),
    IFNULL(rut_cliente,0),
    IFNULL(dv_rut_cliente,' '),
    IFNULL(tipo_operacion,' '),
    IFNULL(operacion,' '),
    IFNULL(sistema,' '),
    IFNULL(segmento,' '),
    IFNULL(criterio_entrada,0),
    IFNULL(origen_deterioro,0),
    IFNULL(fecha_entrada,19000101),
    IFNULL(grupo,' '),
    '{p_periodo_evaluacion}'
FROM
    tmp_tbl_cartdet_crit_ent_crit 
"""  


In [0]:
sql_safe(paso_query310)

##Estadisticas tabla salida

In [0]:
%sql
SELECT
fecha_cierre,
criterio_entrada,
CASE 
  WHEN criterio_entrada=1 THEN 'CLASIFICACION_DETERIORO'
  WHEN criterio_entrada=7 THEN 'MOROSIDAD_NO_HIPCAE'
  WHEN criterio_entrada=8 THEN 'MOROSIDAD_HIPCAE'
  WHEN criterio_entrada=9 THEN 'RENEGOCIADO'
  WHEN criterio_entrada=10 THEN 'REESTRUCTURACION_FORZOSA'
  WHEN criterio_entrada=11 THEN 'LIR'
  WHEN criterio_entrada=12 THEN 'SSFF'
  WHEN criterio_entrada=13 THEN 'FACTORING'
  ELSE 'NO_IDENTIFICADO'    
END                    AS des_criterio_entrada,
COUNT(1) AS CANT_REG
FROM ${bci.dbnamesilver}.tbl_cd_cartdet_crit_ent_crit
GROUP BY 1,2,3
ORDER BY 1,2,3


## Mensaje termino OK

In [0]:
msgerrorx="OK"
dbutils.notebook.exit("{\"coderror\":0, \"msgerror\":\""+msgerrorx+"\"}")